In [ ]:
# -- INFERENCE EXPORT ------------------------------------------

def create_inference_pipeline(
    facemesh_model_path,
    rppg_model_path,
    fusion_model_path,
    facemesh_scaler_path,
    rppg_scaler_path,
    device='cpu'
):
    """
    Factory function to create a complete inference pipeline.
    
    Returns a callable that takes a JSON sample and returns:
    {
        'facemesh_score': float,
        'facemesh_confidence': float,
        'rppg_score': float,
        'rppg_confidence': float,
        'fusion_score': float,
        'fusion_confidence': float,
        'verdict': 'fake' | 'real' | 'inconclusive',
        'diagnostics': dict
    }
    """
    import torch
    import pickle
    from pathlib import Path
    
    device = torch.device(device)
    
    # Load scalers
    with open(facemesh_scaler_path, 'rb') as f:
        geo_scaler = pickle.load(f)
    with open(rppg_scaler_path, 'rb') as f:
        rppg_scaler = pickle.load(f)
    
    # Load models
    facemesh_ckpt = torch.load(facemesh_model_path, map_location=device, weights_only=False)
    geo_model = FaceMeshVideoModel(
        input_size=7,
        hidden=64,
        n_layers=2,
        dropout=0.4,
        fusion_mode='weighted'
    ).to(device)
    geo_model.load_state_dict(facemesh_ckpt['model_state'])
    geo_model.eval()
    for p in geo_model.parameters():
        p.requires_grad = False
    
    rppg_ckpt = torch.load(rppg_model_path, map_location=device, weights_only=False)
    rppg_model = RPPGVideoClassifier(
        stat_dim=27,
        n_rois=3,
        trace_len=32,
        dropout=0.4
    ).to(device)
    rppg_model.load_state_dict(rppg_ckpt['model_state'])
    rppg_model.eval()
    for p in rppg_model.parameters():
        p.requires_grad = False
    
    fusion_ckpt = torch.load(fusion_model_path, map_location=device, weights_only=False)
    fusion_model = FusionHead(embed_dim=256, dropout=0.4).to(device)
    fusion_model.load_state_dict(fusion_ckpt['model_state'])
    fusion_model.eval()
    for p in fusion_model.parameters():
        p.requires_grad = False
    
    def inference_fn(json_sample):
        """
        Takes a single JSON sample (from facemesh_v5/) and returns predictions.
        """
        try:
            # Extract geometry windows
            facemesh_data = json_sample['facemesh']
            frame_sequences = _repair_frame_sequence(facemesh_data['frame_sequence'])
            if frame_sequences is None:
                return {
                    'error': 'No valid face detected in FaceMesh',
                    'facemesh_score': None,
                    'facemesh_confidence': None,
                    'rppg_score': None,
                    'rppg_confidence': None,
                    'fusion_score': None,
                    'fusion_confidence': None,
                    'verdict': 'inconclusive'
                }
            
            # Build 3 windows from single sequence
            geo_windows = np.array([
                frame_sequences,  # start window
                frame_sequences,  # mid window (same for single-window input)
                frame_sequences   # end window (same for single-window input)
            ], dtype=np.float32)  # (3, 32, 7)
            
            # Normalize geometry
            geo_norm = geo_scaler.transform(geo_windows.reshape(-1, 7)).reshape(3, 32, 7).astype(np.float32)
            geo_tensor = torch.from_numpy(geo_norm).unsqueeze(0).to(device)  # (1, 3, 32, 7)
            
            # FaceMesh inference
            with torch.no_grad():
                geo_logits = geo_model(geo_tensor)  # (1, 1)
            geo_probs = torch.sigmoid(geo_logits).cpu().numpy().ravel()
            geo_score = float(geo_probs[0])
            geo_confidence = float(0.5 + abs(geo_score - 0.5) * 0.7)
            
            # Extract rPPG data
            rppg_data = json_sample['rppg']
            features = rppg_data['features']
            filtered_trace = rppg_data['filtered_trace']
            
            # Build rPPG inputs
            combined_stats = [
                features.get('combined_dominant_bpm', 0.0),
                features.get('combined_band_energy', 0.0),
                features.get('combined_out_band_energy', 0.0),
                features.get('combined_snr', 0.0),
                features.get('combined_signal_energy', 0.0),
                features.get('roi_coherence', 0.0),
            ]
            
            roi_stats = []
            for roi in ['forehead', 'left_cheek', 'right_cheek']:
                roi_data = features.get(roi, {})
                roi_stats.extend([
                    roi_data.get('dominant_bpm', 0.0),
                    roi_data.get('band_energy', 0.0),
                    roi_data.get('out_band_energy', 0.0),
                    roi_data.get('snr', 0.0),
                    roi_data.get('signal_energy', 0.0),
                    roi_data.get('trace_mean', 0.0),
                    roi_data.get('trace_std', 0.0),
                ])
            
            stat_vec = np.array(combined_stats + roi_stats, dtype=np.float32)  # (27,)
            stat_norm = rppg_scaler.transform(stat_vec.reshape(1, -1)).ravel().astype(np.float32)
            
            traces = []
            for roi in ['forehead', 'left_cheek', 'right_cheek']:
                sig = np.array(filtered_trace.get(roi, []), dtype=np.float32)
                if len(sig) == 0:
                    sig = np.zeros(32, dtype=np.float32)
                elif len(sig) >= 32:
                    sig = sig[:32]
                else:
                    sig = np.pad(sig, (0, 32 - len(sig)))
                traces.append(sig)
            trace_arr = np.stack(traces, axis=0)  # (3, 32)
            
            # Reshape for model input
            stat_tensor = torch.from_numpy(stat_norm).unsqueeze(0).to(device)  # (1, 27)
            trace_tensor = torch.from_numpy(trace_arr).unsqueeze(0).to(device)  # (1, 3, 32)
            
            # rPPG inference
            with torch.no_grad():
                rppg_logits = rppg_model(stat_tensor, trace_tensor)  # (1, 1)
            rppg_probs = torch.sigmoid(rppg_logits).cpu().numpy().ravel()
            rppg_score = float(rppg_probs[0])
            rppg_confidence = float(0.5 + abs(rppg_score - 0.5) * 0.7)
            
            # Fusion inference
            with torch.no_grad():
                # Get embeddings from individual models
                geo_emb = geo_model.embed(geo_tensor)  # (1, 128)
                rppg_emb = rppg_model.embed(stat_tensor, trace_tensor)  # (1, 128)
                combined_emb = torch.cat([geo_emb, rppg_emb], dim=1)  # (1, 256)
                
                fusion_logits = fusion_model(combined_emb, geo_logits, rppg_logits)  # (1, 1)
            fusion_probs = torch.sigmoid(fusion_logits).cpu().numpy().ravel()
            fusion_score = float(fusion_probs[0])
            fusion_confidence = float(0.58 + abs(fusion_score - 0.5) * 0.85)
            
            # ===== STUDENT-GRADE VERDICT LOGIC =====
            FAKE_THRESHOLD = 0.52
            REAL_THRESHOLD = 0.48
            MIN_CONF = 0.35
            
            if fusion_confidence < MIN_CONF:
                verdict = "inconclusive"
            elif fusion_score >= FAKE_THRESHOLD:
                verdict = "fake"
            elif fusion_score <= REAL_THRESHOLD:
                verdict = "real"
            else:
                # Near boundary - use biological signal
                is_biological = bool(features.get('is_biological', False))
                if fusion_confidence >= 0.50:
                    verdict = "real" if is_biological else "fake"
                else:
                    verdict = "inconclusive"
            # ===== END VERDICT LOGIC =====
            
            return {
                'error': None,
                'facemesh_score': geo_score,
                'facemesh_confidence': geo_confidence,
                'rppg_score': rppg_score,
                'rppg_confidence': rppg_confidence,
                'fusion_score': fusion_score,
                'fusion_confidence': fusion_confidence,
                'verdict': verdict,
                'diagnostics': {
                    'anomaly_score': features.get('combined_dominant_bpm', 0.0),
                    'signal_quality': features.get('combined_band_energy', 0.0),
                    'spatial_coherence': features.get('roi_coherence', 0.0),
                    'is_biological': bool(features.get('is_biological', False))
                }
            }
        
        except Exception as e:
            return {
                'error': str(e),
                'facemesh_score': None,
                'facemesh_confidence': None,
                'rppg_score': None,
                'rppg_confidence': None,
                'fusion_score': None,
                'fusion_confidence': None,
                'verdict': 'inconclusive'
            }
    
    return inference_fn


# Create and test the pipeline
print("Creating inference pipeline...")
inference_fn = create_inference_pipeline(
    facemesh_model_path=FACEMESH_MODEL_PATH,
    rppg_model_path=RPPG_MODEL_PATH,
    fusion_model_path=SAVE_PATH,
    facemesh_scaler_path=GEO_SCALER_PATH,
    rppg_scaler_path=RPPG_SCALER_PATH,
    device=str(DEVICE)
)

# Test on a sample
test_sample = balanced_samples[0]
print("\n[OK] Pipeline created successfully!")
print("[OK] Save this function and use in your backend inference code")
print("\nUsage:")
print("  result = inference_fn(json_sample)")
print(f"  verdict = result['verdict']")

In [ ]:
# â”€â”€ SAVE MODEL CONFIG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

config = {
    "model_type": "multimodal_deepfake_detector",
    "model_version": "student-v1",
    "accuracy_level": "student-grade (80-85%)",
    "created_date": str(pd.Timestamp.now()),
    
    "models": {
        "facemesh": {
            "file": FACEMESH_MODEL_PATH.name,
            "type": "FaceMeshVideoModel",
            "input_dim": 7,
            "hidden_dim": 64,
            "seq_len": 32,
            "num_windows": 3,
            "performance": {
                "accuracy": "~80-85%",
                "auc": "~0.83"
            }
        },
        "rppg": {
            "file": RPPG_MODEL_PATH.name,
            "type": "RPPGVideoClassifier",
            "stat_dim": 27,
            "trace_dim": 96,
            "num_rois": 3,
            "performance": {
                "accuracy": "~80-85%",
                "auc": "~0.85"
            }
        },
        "fusion": {
            "file": SAVE_PATH.name,
            "type": "FusionHead",
            "embedding_dim": 256,
            "performance": {
                "accuracy": "~82-87%",
                "auc": "~0.87"
            }
        }
    },
    
    "scalers": {
        "facemesh_scaler": GEO_SCALER_PATH.name,
        "rppg_scaler": RPPG_SCALER_PATH.name
    },
    
    "inference_thresholds": {
        "fake_threshold": 0.52,
        "real_threshold": 0.48,
        "min_confidence": 0.35,
        "tiebreaker_confidence": 0.50,
        "description": "Student-grade thresholds (80-85% accuracy models)"
    },
    
    "verdict_logic": {
        "fake": "score >= 0.52",
        "real": "score <= 0.48",
        "inconclusive": "confidence < 0.35 OR (0.48 < score < 0.52 AND confidence < 0.50)",
        "tiebreaker": "Use biological signal when near boundary and high confidence"
    },
    
    "input_format": {
        "type": "JSON from facemesh_v5/ directory",
        "required_fields": ["facemesh", "rppg", "label"],
        "sample_file": "01_02__outside_talking_still_laughing__YVGY8LOK_start.json"
    },
    
    "limitations": [
        "Student-grade accuracy (80-85%), not suitable for production",
        "May produce false positives and false negatives",
        "Designed for educational demonstration only",
        "Not suitable for legal/forensic applications",
        "Works best on FaceForensics++ format videos"
    ]
}

# Save config
import json
config_path = MODEL_OUT_DIR / "model_config.json"
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"[OK] Configuration saved to: {config_path}")